In [5]:
import pandas as pd
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [7]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [8]:
y = df['Outcome']
x = df.drop(columns = ['Outcome'])

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

scale = StandardScaler()
x_train = scale.fit_transform(x_train)
x_test = scale.transform(x_test)


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, x_train, y_train, cv=3, scoring='accuracy').mean()

    return score 

In [15]:
import optuna
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)

[I 2026-08-07 23:18:22,428] A new study created in memory with name: no-name-64f7b876-33a0-43e2-ab86-ec1f9ef093a2
[I 2026-08-07 23:18:22,828] Trial 0 finished with value: 0.7638291088793241 and parameters: {'n_estimators': 74, 'max_depth': 8}. Best is trial 0 with value: 0.7638291088793241.
[I 2026-08-07 23:18:23,865] Trial 1 finished with value: 0.7736011477761836 and parameters: {'n_estimators': 195, 'max_depth': 9}. Best is trial 1 with value: 0.7736011477761836.
[I 2026-08-07 23:18:24,430] Trial 2 finished with value: 0.7703650565917424 and parameters: {'n_estimators': 110, 'max_depth': 6}. Best is trial 1 with value: 0.7736011477761836.
[I 2026-08-07 23:18:25,034] Trial 3 finished with value: 0.7670970827355332 and parameters: {'n_estimators': 114, 'max_depth': 8}. Best is trial 1 with value: 0.7736011477761836.
[I 2026-08-07 23:18:25,897] Trial 4 finished with value: 0.7703331739199745 and parameters: {'n_estimators': 157, 'max_depth': 19}. Best is trial 1 with value: 0.773601147